# Exemple d'Utilisation - Pipeline MLOps

## 1. Importer les modules

In [1]:
#import sys
#sys.path.append(r"C:\_sources\Jenedai\jefraudai\src")

from fraud_detection.pipelines.training_pipeline import MLPipeline
from fraud_detection.data.data_loader import load_config
import logging
logging.basicConfig(level=logging.DEBUG, format='%(asctime)s - %(levelname)s - %(message)s')

In [2]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(".env"), override=True)      # config
load_dotenv(find_dotenv(".env.secrets"), override=True)   # secrets only useful on local environment

True

## 2. Initialiser le pipeline

In [3]:
print("\n>>> Initialisation du pipeline...\n")
pipeline = MLPipeline(config_path="../src/configs/config.yaml")
#file_path = "data/raw/extract_csv_engie_dataset.csv"
#file_path = "data/raw/extract_csv_enedis_dataset_500000.csv"
file_path = "../data/raw/fraudTest.csv"

INFO:fraud_detection.pipelines.training_pipeline:Pipeline MLOps initialisé avec model_name=fraud_detection_model



>>> Initialisation du pipeline...



## 3. Exécuter le pipeline complet

In [4]:
# Note: Remplacer par votre fichier de données
#print("\n>>> Lancement du pipeline complet...\n")
#success = pipeline.run_full_pipeline(file_path)

#if success:
#    print("\n✓ Pipeline exécuté avec succès!")
#    print(f"\nMétriques finales: {pipeline.metrics}")
#else:
#    print("\n✗ Erreur lors de l'exécution du pipeline")

## 4. Alternative: Exécuter étape par étape

In [5]:
print("\n" + "="*50)
print("EXÉCUTION ÉTAPE PAR ÉTAPE (alternative)")
print("="*50)

pipeline2 = pipeline  # Cloner le pipeline pour une exécution étape par étape


EXÉCUTION ÉTAPE PAR ÉTAPE (alternative)


In [6]:
print("\n1. Chargement des données...")
pipeline2.step_1_load_data(file_path)

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 1: CHARGEMENT DES DONNÉES ===TE



1. Chargement des données...
Données chargées avec succès_: ../data/raw/fraudTest.csv
Forme des données: (555719, 23)
Colonnes: ['Unnamed: 0', 'trans_date_trans_time', 'cc_num', 'merchant', 'category', 'amt', 'first', 'last', 'gender', 'street', 'city', 'state', 'zip', 'lat', 'long', 'city_pop', 'job', 'dob', 'trans_num', 'unix_time', 'merch_lat', 'merch_long', 'is_fraud\r']
Types: {'Unnamed: 0': dtype('int64'), 'trans_date_trans_time': dtype('O'), 'cc_num': dtype('int64'), 'merchant': dtype('O'), 'category': dtype('O'), 'amt': dtype('float64'), 'first': dtype('O'), 'last': dtype('O'), 'gender': dtype('O'), 'street': dtype('O'), 'city': dtype('O'), 'state': dtype('O'), 'zip': dtype('int64'), 'lat': dtype('float64'), 'long': dtype('float64'), 'city_pop': dtype('int64'), 'job': dtype('O'), 'dob': dtype('O'), 'trans_num': dtype('O'), 'unix_time': dtype('int64'), 'merch_lat': dtype('float64'), 'merch_long': dtype('float64'), 'is_fraud\r': dtype('int64')}
Head:    Unnamed: 0 trans_date_tra

True

In [7]:
print("\n2. Validation des données...")
pipeline2.step_2_validate_data(save_report=True)

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 2: VALIDATION DES DONNÉES ===



2. Validation des données...


INFO:fraud_detection.data.data_validator:Validation: Valeurs manquantes=0.00%, Doublons=0.00%
INFO:fraud_detection.pipelines.training_pipeline:Résultats de validation: True
INFO:fraud_detection.data.data_validator:Rapport simple créé: outputs/evidently_report.html


np.True_

In [8]:
print("\n3. Préparation et Prétraitement des données...")
print("   - Split train/test")
print("   - Détection automatique des types de colonnes")
print("   - Imputation des valeurs manquantes")
print("   - Normalisation des données numériques")
print("   - Encodage des variables catégories")

pipeline2.step_3_prepare_data()

if pipeline2.preprocessor and pipeline2.feature_names:
    print(f"\n✓ Données préparées et prétraitées!")
    print(f"  Nombre de features après preprocessing: {len(pipeline2.feature_names)}")
    print(f"  Forme train: {pipeline2.X_train.shape}")
    print(f"  Forme test: {pipeline2.X_test.shape}")

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 3: TRANSFORMATION DES DONNÉES ===
INFO:fraud_detection.data.data_transformer:Détection de colonnes sans nom: ['Unnamed: 0']



3. Préparation et Prétraitement des données...
   - Split train/test
   - Détection automatique des types de colonnes
   - Imputation des valeurs manquantes
   - Normalisation des données numériques
   - Encodage des variables catégories
=== ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT (stateless)===


INFO:fraud_detection.data.data_transformer:Supprimé 15 colonne(s): ['Unnamed: 0', 'cc_num', 'merchant', 'first', 'last', 'street', 'trans_num', 'unix_time', 'dob', 'city', 'state', 'lat', 'long', 'merch_lat', 'merch_long']
INFO:fraud_detection.data.data_transformer:Nettoyage des données terminé
INFO:fraud_detection.pipelines.training_pipeline:✓ Données nettoyées et transformées
INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 3: PRÉPARATION ET PRÉTRAITEMENT (stateful)===
INFO:fraud_detection.pipelines.training_pipeline:  → Division train/test...


  ℹ Colonne cible (config): is_fraud
Données après nettoyage: (555719, 11)
Aperçu des premières lignes de data_clean:
        category    amt gender    zip  city_pop                     job  \
0  personal_care   2.86      M  29209    333497     Mechanical engineer   
1  personal_care  29.84      F  84002       302  Sales professional, IT   

   is_fraud  trans_date_trans_time_month  trans_date_trans_time_hour  \
0         0                            6                          12   
1         0                            6                          12   

   trans_date_trans_time_weekday  trans_date_trans_time_is_weekend  
0                              6                                 1  
1                              6                                 1  
Colonne cible utilisée (paramètre): is_fraud
Features shape: (555719, 10), Target shape: (555719,)
Aperçu des premières lignes de X:
        category    amt gender    zip  city_pop                     job  \
0  personal_care   2.86 

INFO:fraud_detection.pipelines.training_pipeline:  ✓ Split effectué: train=(444575, 10), test=(111144, 10)
INFO:fraud_detection.pipelines.training_pipeline:  → Prétraitement AutoGluon désactivé (mode brut, AutoGluon gère lui-même le preprocessing)
INFO:fraud_detection.data.data_preparation:=== PRÉPARATION DES DONNÉES ===
INFO:fraud_detection.data.data_preparation:
1️⃣ Détection des types de features...________
INFO:fraud_detection.data.data_preparation:🔍 Features numériques détectées: ['amt', 'zip', 'city_pop', 'trans_date_trans_time_month', 'trans_date_trans_time_hour', 'trans_date_trans_time_weekday', 'trans_date_trans_time_is_weekend']
INFO:fraud_detection.data.data_preparation:🔍 Features catégories détectées: ['category', 'gender', 'job']
INFO:fraud_detection.data.data_preparation:
2️⃣ AutoGluon mode — prétraitement minimal...
INFO:fraud_detection.data.data_preparation:
✓ Préparation AutoGluon terminée

INFO:fraud_detection.pipelines.training_pipeline:  ✓ Prétraitement effectué: (4

In [9]:
print("\n4. Entraînement du modèle...")
print("   (sur les données préparées/préprocessées)")
print(set(pipeline2.y_train))
pipeline2.step_4_train_model()


if pipeline2.model:
    print(f"✓ Modèle entraîné: {type(pipeline2.model).__name__}")

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 4: ENTRAÎNEMENT DU MODÈLE ===
INFO:fraud_detection.models.model:X_train shape: (444575, 10), y_train shape: (444575,)



4. Entraînement du modèle...
   (sur les données préparées/préprocessées)
{0, 1}


c:\Users\SustCoop\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Local\pypoetry\Cache\virtualenvs\fraud-detection-aIYKtiDm-py3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No path specified. Models will be saved in: "AutogluonModels\ag-20260601_200150"
Preset alias specified: 'good' maps to 'good_quality'.
Verbosity: 2 (Standard Logging)
	Failed to import torch or check CUDA availability!Please ensure you have the correct version of PyTorch installed by running `pip install -U torch`
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          4
Pytorch Version:    Can't import torch
CUDA Version:       Can't get cuda versio

✓ Modèle entraîné: TabularPredictor


In [10]:
print("\n5. Évaluation du modèle...")
pipeline2.step_5_evaluate_model()

if pipeline2.metrics:
    print("\n--- RÉSULTATS ---")
    for key, value in pipeline2.metrics.items():
        print(f"  {key}: {value:.4f}")

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 5: ÉVALUATION DU MODÈLE ===



5. Évaluation du modèle...


INFO:fraud_detection.models.model:Autogluon - Métriques: {'accuracy': 0.9979666018858417, 'balanced_accuracy': np.float64(0.8165852900237096), 'mcc': 0.7084634752704152, 'roc_auc': np.float64(0.9959055879137007), 'f1': 0.7049608355091384, 'precision': 0.7941176470588235, 'recall': 0.6338028169014085, 'accuracy_manual': 0.9979666018858417, 'precision_manual': 0.7941176470588235, 'recall_manual': 0.6338028169014085, 'f1_manual': 0.7049608355091384}



--- RÉSULTATS ---
  accuracy: 0.9980
  balanced_accuracy: 0.8166
  mcc: 0.7085
  roc_auc: 0.9959
  f1: 0.7050
  precision: 0.7941
  recall: 0.6338
  accuracy_manual: 0.9980
  precision_manual: 0.7941
  recall_manual: 0.6338
  f1_manual: 0.7050


In [11]:
print("\n6. Monitoring de la performance...")
pipeline2.step_6_monitor_performance()

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 6: MONITORING DE LA PERFORMANCE ===



6. Monitoring de la performance...


INFO:fraud_detection.monitoring.performance_monitor:Métriques de performance calculées: {'accuracy': 0.9999077770904796, 'balanced_accuracy': 0.9909718393748681}
INFO:fraud_detection.monitoring.performance_monitor:Métriques de performance calculées: {'accuracy': 0.9979666018858417, 'balanced_accuracy': 0.8165852900237096}
INFO:fraud_detection.pipelines.training_pipeline:Résultats du monitoring: Drift=True
INFO:fraud_detection.pipelines.training_pipeline:Performance test: {'accuracy': 0.9979666018858417, 'balanced_accuracy': 0.8165852900237096}


True

In [12]:
print("\n7. Logging avec MLflow...")
pipeline2.step_7_log_with_mlflow()

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 7: LOGGING AVEC MLFLOW ===



7. Logging avec MLflow...


INFO:fraud_detection.models.mlflow_tracker:Tracking URI configuré: https://jefraudai-mlflow.hf.space/
INFO:fraud_detection.models.mlflow_tracker:Répertoire local des artefacts configuré: c:\_sources\Jenedai\jefraudai\notebooks\jefraudai\mlflow
INFO:fraud_detection.models.mlflow_tracker:Run MLflow démarrée - Expérience: Fraud_Detection
INFO:fraud_detection.models.mlflow_tracker:Paramètres enregistrés: {'test_size': 0.2, 'random_state': 42, 'model_type': 'auto_gluon'}
INFO:fraud_detection.models.mlflow_tracker:Métriques enregistrées: ['accuracy', 'balanced_accuracy', 'mcc', 'roc_auc', 'f1', 'precision', 'recall', 'accuracy_manual', 'precision_manual', 'recall_manual', 'f1_manual', 'monitoring_drift_mean_drift', 'monitoring_drift_std_drift', 'monitoring_drift_drift_detected', 'monitoring_test_accuracy', 'monitoring_test_balanced_accuracy', 'monitoring_train_accuracy', 'monitoring_train_balanced_accuracy']
INFO:fraud_detection.models.mlflow_tracker:Chemin AutoGluon détecté: c:\_sources\Jen

🏃 View run bittersweet-crow-683 at: https://jefraudai-mlflow.hf.space/#/experiments/1/runs/3f465cd6dda545a4be68a4a5e8963b06
🧪 View experiment at: https://jefraudai-mlflow.hf.space/#/experiments/1


INFO:fraud_detection.models.mlflow_tracker:Run MLflow terminée
INFO:fraud_detection.models.mlflow_tracker:Session d'entraînement enregistrée dans MLflow


True

In [16]:
# Step interessante mais non fonctionnel (Refactoring necessaire mais non prioritaire)
print("\n8. Gestion des Stages du Modèle MLflow...")
# Métriques pour prédiction énergétique (5 jours)
# MAE = Mean Absolute Error (valeur absolue moyenne des erreurs)
# RMSE = Root Mean Squared Error (racine carrée de la moyenne des carrés)
# Accuracy = Pour certains modèles de classification
metric_keys = ["mae", "rmse", "accuracy"]
pipeline2.step_8_manage_model_stages(metric_keys=metric_keys, min_improvement=0.0)

INFO:fraud_detection.pipelines.training_pipeline:=== ÉTAPE 8: GESTION DES ALIASES DU MODÈLE ===
INFO:fraud_detection.models.mlflow_tracker:Tracking URI configuré: https://jefraudai-mlflow.hf.space/
INFO:fraud_detection.models.mlflow_tracker:Répertoire local des artefacts configuré: C:\_sources\Jenedai\jefraudai\jefraudai\mlflow



8. Gestion des Stages du Modèle MLflow...


INFO:fraud_detection.pipelines.training_pipeline:
[1/4] Récupération du run_id et enregistrement du modèle en Staging...
INFO:fraud_detection.pipelines.training_pipeline:  Métriques à comparer (priorité): ['mae', 'rmse', 'accuracy']
INFO:fraud_detection.pipelines.training_pipeline:  ℹ Run trouvée: 3f465cd6dda545a4be68a4a5e8963b06
ERROR:fraud_detection.models.mlflow_tracker:Erreur lors de l'enregistrement du modèle: RESOURCE_DOES_NOT_EXIST: Registered Model with name=fraud_detection_model not found
ERROR:fraud_detection.pipelines.training_pipeline:Impossible d'enregistrer la version du modèle


False

In [14]:
print("\n✓ Pipeline étape par étape terminé!")


✓ Pipeline étape par étape terminé!
